# 🚀 L&D Designs — Automated Email & SMS Outreach

Sends cold outreach **emails** (via Gmail) and **SMS** (via Twilio) to leads from your multi-type spreadsheet.

---

## Before you run anything:

**Gmail App Password:**
1. Go to [myaccount.google.com/security](https://myaccount.google.com/security)
2. Make sure 2-Step Verification is ON
3. Click **App passwords** → name it anything → click **Create**
4. Copy the 16-character password (e.g. `abcd efgh ijkl mnop`)

**Twilio (for SMS):**
- Sign up free at [twilio.com](https://www.twilio.com) — free trial includes £15 credit
- Get your Account SID, Auth Token, and a free Twilio phone number
- *SMS is optional — leave TWILIO fields blank to email only*

---
**Run cells in order: 1 → 2 → 3 → 4**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'twilio'])
print('✅ Dependencies ready.')

In [ ]:
# ── Cell 2: YOUR DETAILS — fill in everything here ────────────────────────────

# Gmail
GMAIL_ADDRESS      = 'your@gmail.com'            # your Gmail address
GMAIL_APP_PASSWORD = 'xxxx xxxx xxxx xxxx'        # 16-char app password

# Twilio SMS (optional — leave blank to skip SMS)
TWILIO_ACCOUNT_SID = ''   # e.g. 'AC1234...'
TWILIO_AUTH_TOKEN  = ''   # e.g. '1a2b3c...'
TWILIO_FROM_NUMBER = ''   # e.g. '+441234567890'

# Your info
YOUR_NAME   = 'Dylan'
YOUR_PHONE  = '07504683058'
SHOWCASE_LINK = 'https://mellow-speculoos-d1850c.netlify.app/showcase.html'

# How many to send per run (to avoid spam flags — start small)
MAX_EMAILS = 30
MAX_SMS    = 20

# ─────────────────────────────────────────────────────────────────────────────
USE_SMS = bool(TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN and TWILIO_FROM_NUMBER)
USE_EMAIL = GMAIL_ADDRESS != 'your@gmail.com' and GMAIL_APP_PASSWORD != 'xxxx xxxx xxxx xxxx'

print('Email sending:', '✅ Configured' if USE_EMAIL else '❌ Not configured — fill in GMAIL_ADDRESS and GMAIL_APP_PASSWORD')
print('SMS sending  :', '✅ Configured' if USE_SMS else '⚪ Not configured (optional)')
if not USE_EMAIL:
    print()
    print('👆 Fill in your Gmail details above, then re-run this cell.')

In [ ]:
# ── Cell 3: Upload leads spreadsheet ─────────────────────────────────────────
import re
import openpyxl
from google.colab import files

print('Select your multi_leads_*.xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb.active
headers = [str(c.value or '').strip() for c in ws[1]]
print('Columns:', headers)

all_leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0]:
        all_leads.append(dict(zip(headers, row)))
print('Total rows:', len(all_leads))

def gf(lead, *keys):
    for k in keys:
        v = lead.get(k)
        if v and str(v).strip().lower() not in ('', 'none', 'nan', 'null'):
            return str(v).strip()
    return ''

def clean_phone(p):
    d = re.sub(r'[^\d+]', '', p)
    if d.startswith('0'): d = '+44' + d[1:]
    elif d and not d.startswith('+'): d = '+44' + d
    return d

# Filter: no website only
email_leads = []
sms_leads   = []
for l in all_leads:
    status = gf(l, 'Website Status', 'website_status').upper()
    if 'ACTIVE' in status:
        continue
    email = gf(l, 'Email', 'email', 'Email Address')
    phone = gf(l, 'Phone', 'phone', 'Phone Number')
    if email and '@' in email:
        email_leads.append(l)
    elif phone:  # SMS only if no email
        sms_leads.append(l)

print()
print(f'Ready to email : {len(email_leads)}')
print(f'Ready to SMS   : {len(sms_leads)} (no email, has phone)')
print(f'Skipped        : {len(all_leads) - len(email_leads) - len(sms_leads)} (has website or no contact)')
print()
print(f'Will send up to {MAX_EMAILS} emails and {MAX_SMS} SMS this run.')

In [ ]:
# ── Cell 4: Send emails & SMS ─────────────────────────────────────────────────
import smtplib, time, random, csv
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Verify config
if not USE_EMAIL:
    print('❌ Fill in GMAIL_ADDRESS and GMAIL_APP_PASSWORD in Cell 2 first.')
    raise SystemExit

EMAIL_SUBJECT = 'Quick question — website for {name}'
EMAIL_BODY = """Hi,

I noticed {name} doesn't have a website yet.

I build professional websites for local businesses in Wigan from just £199 — usually done within a week. No monthly fees, one-off payment.

Here's some of my work: {showcase}

Would you be interested in a free quote?

{your_name}
L&D Designs
WhatsApp / Call: {your_phone}"""

SMS_BODY = "Hi, I noticed {name} doesn't have a website. I build professional sites for Wigan businesses from £199 — done in days, no monthly fees. Free quote? - {your_name}, L&D Designs"

log = []
sent_email = 0
sent_sms   = 0
failed     = 0

# ── EMAIL ─────────────────────────────────────────────────────────────────────
batch = email_leads[:MAX_EMAILS]
if batch:
    print(f'Connecting to Gmail SMTP...')
    try:
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        print('✅ Connected.')
    except Exception as e:
        print(f'❌ Gmail connection failed: {e}')
        print('Check your GMAIL_ADDRESS and GMAIL_APP_PASSWORD in Cell 2.')
        raise

    print(f'\nSending {len(batch)} emails...\n')
    for i, lead in enumerate(batch):
        name  = gf(lead, 'Business Name', 'name') or 'there'
        email = gf(lead, 'Email', 'email', 'Email Address')
        ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        subject = EMAIL_SUBJECT.format(name=name)
        body    = EMAIL_BODY.format(name=name, showcase=SHOWCASE_LINK,
                                    your_name=YOUR_NAME, your_phone=YOUR_PHONE)
        msg = MIMEMultipart()
        msg['From']    = GMAIL_ADDRESS
        msg['To']      = email
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain'))

        try:
            server.sendmail(GMAIL_ADDRESS, email, msg.as_string())
            sent_email += 1
            status = 'sent'
            print(f'[{i+1}/{len(batch)}] ✅ {name} → {email}')
        except Exception as e:
            failed += 1
            status = f'failed: {e}'
            print(f'[{i+1}/{len(batch)}] ❌ {name} → {email} — {e}')

        log.append({'type': 'email', 'name': name, 'contact': email,
                    'status': status, 'timestamp': ts})

        if i < len(batch) - 1 and status == 'sent':
            time.sleep(random.uniform(3, 6))

    server.quit()
    print(f'\nEmail batch done. Sent: {sent_email}  Failed: {failed}')

# ── SMS (Twilio) ──────────────────────────────────────────────────────────────
if USE_SMS and sms_leads:
    from twilio.rest import Client as TwilioClient
    tw = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
    sms_batch = sms_leads[:MAX_SMS]
    print(f'\nSending {len(sms_batch)} SMS messages...\n')

    for i, lead in enumerate(sms_batch):
        name  = gf(lead, 'Business Name', 'name') or 'there'
        phone = clean_phone(gf(lead, 'Phone', 'phone', 'Phone Number'))
        ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        body  = SMS_BODY.format(name=name, your_name=YOUR_NAME)

        try:
            msg = tw.messages.create(body=body, from_=TWILIO_FROM_NUMBER, to=phone)
            sent_sms += 1
            status = 'sent'
            print(f'[{i+1}/{len(sms_batch)}] ✅ {name} → {phone}')
        except Exception as e:
            failed += 1
            status = f'failed: {e}'
            print(f'[{i+1}/{len(sms_batch)}] ❌ {name} → {phone} — {e}')

        log.append({'type': 'sms', 'name': name, 'contact': phone,
                    'status': status, 'timestamp': ts})
        time.sleep(1)

    print(f'\nSMS batch done. Sent: {sent_sms}')

# ── Save log ──────────────────────────────────────────────────────────────────
OUTREACH_LOG = log
log_file = f'outreach_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
with open(log_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['type','name','contact','status','timestamp'])
    writer.writeheader()
    writer.writerows(log)

print(f'\n=========================================')
print(f'TOTAL: {sent_email} emails + {sent_sms} SMS sent')
print(f'Log saved as {log_file}')
print(f'Run Cell 5 to download the log.')

In [ ]:
# ── Cell 5: Download log ──────────────────────────────────────────────────────
from google.colab import files
try:
    files.download(log_file)
    print(f'Downloading {log_file} — check your Downloads folder.')
    print()
    print(f'Sent: {sent_email} emails, {sent_sms} SMS')
    success = sum(1 for r in OUTREACH_LOG if r["status"] == "sent")
    print(f'Success rate: {success}/{len(OUTREACH_LOG)}')
except NameError:
    print('Run Cell 4 first to send emails.')